# 🧩 Python, NumPy & Pandas — The Data Backbone of ML

Every machine learning model — from a simple linear regressor to a transformer — ultimately consumes **numbers arranged in arrays**. Before we touch any algorithm, we need to be fluent in the two libraries that hold ML's entire data layer together:

- **NumPy** → fast, vectorized numerical arrays (the "tensor" before deep learning made that word popular)
- **Pandas** → labeled, tabular data (the "spreadsheet" that talks to Python)

This notebook builds intuition + math for both, then ties them together in a mini end-to-end example.

📖 Full mathematical explanation: [README.md](README.md)


## 1. Why NumPy? — Vectorization vs Loops

A Python `list` stores **pointers to objects** scattered in memory. A NumPy `ndarray` stores **raw numbers in one contiguous memory block**, so operations run as compiled C loops (SIMD-vectorized) instead of Python's slow bytecode loop.

Mathematically, for two vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^n$, element-wise addition is:

$$\mathbf{c}_i = \mathbf{a}_i + \mathbf{b}_i \quad \forall i \in \{1, ..., n\}$$

NumPy computes all $n$ additions in one vectorized instruction — no explicit Python `for` loop needed.


In [ ]:
import numpy as np
import pandas as pd
import time

# --- Speed comparison: pure Python loop vs NumPy vectorized op ---
n = 1_000_000
py_list_a = list(range(n))
py_list_b = list(range(n))
np_arr_a = np.arange(n)
np_arr_b = np.arange(n)

# Pure Python: element-wise add via list comprehension (interpreted, one object at a time)
start = time.time()
py_result = [a + b for a, b in zip(py_list_a, py_list_b)]
py_time = time.time() - start

# NumPy: same operation, vectorized in compiled C
start = time.time()
np_result = np_arr_a + np_arr_b
np_time = time.time() - start

print(f"Pure Python loop : {py_time:.4f} sec")
print(f"NumPy vectorized : {np_time:.4f} sec")
print(f"Speedup          : {py_time / np_time:.1f}x faster")


## 2. Array Creation & Attributes

Every `ndarray` has three attributes that matter constantly in ML code:

| Attribute | Meaning | ML relevance |
|---|---|---|
| `shape` | dimensions, e.g. `(batch, features)` | must match between layers/operations |
| `dtype` | data type, e.g. `float32` | controls memory + precision (deep learning defaults to `float32`) |
| `ndim` | number of axes | scalar=0, vector=1, matrix=2, tensor=3+ |


In [ ]:
# Common creation patterns you'll reuse constantly in ML pipelines
zeros = np.zeros((3, 4))          # placeholder tensor, e.g. initializing an output buffer
ones = np.ones((2, 2))            # useful for bias initialization or masks
identity = np.eye(3)              # identity matrix I — used in regularization (Ridge: X^T X + λI)
ranged = np.arange(0, 10, 2)      # like Python range(), but returns an array
linear = np.linspace(0, 1, 5)     # 5 evenly spaced points in [0, 1] — great for probability grids
random_uniform = np.random.rand(2, 3)     # uniform [0, 1) — common weight init baseline

sample = np.array([[1, 2, 3], [4, 5, 6]])
print("shape:", sample.shape)   # (2 rows, 3 cols) -> (batch_size=2, features=3) in ML terms
print("dtype:", sample.dtype)
print("ndim :", sample.ndim)
print("size :", sample.size)    # total elements = shape product


## 3. Indexing, Slicing & Boolean Masking

NumPy indexing generalizes Python's `list[start:stop:step]` to N dimensions, and adds **boolean masking** — the mechanism behind almost every "filter my dataset" operation in ML preprocessing.

Given condition $f(x)$, boolean masking selects the subset:

$$S = \{ x_i \mid f(x_i) = \text{True} \}$$


In [ ]:
matrix = np.arange(1, 13).reshape(3, 4)   # reshape flat 12 elements into 3x4
print("Full matrix:\n", matrix)

print("\nRow 1:            ", matrix[1])          # second row
print("Column 2:         ", matrix[:, 2])          # third column, all rows
print("Sub-block (top-left 2x2):\n", matrix[:2, :2])

# Boolean masking — the core operation behind "filter rows where condition is true"
mask = matrix > 6
print("\nMask (matrix > 6):\n", mask)
print("Values where matrix > 6:", matrix[mask])     # flattened array of matching values

# Fancy indexing: select specific rows by index list
print("\nRows [0, 2]:\n", matrix[[0, 2]])


## 4. Broadcasting — Operating on Mismatched Shapes

Broadcasting lets NumPy apply operations between arrays of **different shapes** without manually copying data, following two rules:

1. Compare shapes from the **rightmost** dimension.
2. Two dimensions are compatible if they are **equal**, or one of them is **1**.

This is exactly how you normalize a feature matrix $X \in \mathbb{R}^{n \times d}$ by a per-feature mean vector $\mu \in \mathbb{R}^{d}$:

$$X_{\text{norm}}[i, j] = X[i, j] - \mu[j]$$

without writing an explicit loop over rows $i$.


In [ ]:
X = np.array([[10, 200, 3000],
              [20, 400, 6000],
              [30, 600, 9000]], dtype=float)   # 3 samples, 3 features

mean = X.mean(axis=0)   # axis=0 -> collapse rows, one mean PER COLUMN (per feature)
std = X.std(axis=0)

print("Per-feature mean:", mean)
print("Per-feature std :", std)

# Broadcasting: (3,3) array minus (3,) vector -> vector is "stretched" across every row
X_standardized = (X - mean) / std
print("\nStandardized X (zero mean, unit variance per feature):\n", X_standardized)


## 5. Aggregation with `axis` — The Most Confused Concept in NumPy

`axis=0` collapses **rows** (result has one value per column). `axis=1` collapses **columns** (result has one value per row). Think of `axis` as **"which dimension disappears."**

For a matrix $X \in \mathbb{R}^{n \times d}$:

$$\text{mean}(X, \text{axis}=0)_j = \frac{1}{n}\sum_{i=1}^{n} X_{ij} \quad \text{(per-feature mean, used in normalization)}$$
$$\text{mean}(X, \text{axis}=1)_i = \frac{1}{d}\sum_{j=1}^{d} X_{ij} \quad \text{(per-sample mean, e.g. average pixel intensity)}$$


In [ ]:
print("Column-wise sum (axis=0):", X.sum(axis=0))   # one sum per feature/column
print("Row-wise sum    (axis=1):", X.sum(axis=1))   # one sum per sample/row

# Linear algebra: dot product and matrix multiplication — the core op of every neural network layer
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("\nDot product a·b:", np.dot(a, b))           # scalar: sum(a_i * b_i)

W = np.random.randn(3, 2)   # weight matrix, e.g. mapping 3 input features -> 2 outputs
print("\nMatrix multiply X @ W (forward pass style):\n", X @ W)


## 6. Random Numbers — Reproducibility Matters in ML

`np.random.seed()` fixes the pseudo-random generator's starting state, so every run of your notebook produces **identical** train/test splits, weight initializations, and shuffle orders — critical for debugging and fair experiment comparison.


In [ ]:
np.random.seed(42)   # fixing the seed = reproducible experiments

# Simulating a train/test split index shuffle (the mechanism behind train_test_split)
indices = np.arange(10)
np.random.shuffle(indices)
print("Shuffled indices:", indices)

split_point = int(0.8 * len(indices))
train_idx, test_idx = indices[:split_point], indices[split_point:]
print("Train indices:", train_idx)
print("Test indices :", test_idx)


## 7. Pandas — Labeled Data on Top of NumPy

A Pandas **`Series`** is a NumPy array + an index (labels). A **`DataFrame`** is a dict of aligned `Series` — conceptually a table where every column is its own labeled array.

$$\text{DataFrame} = \{ \text{col}_1: \text{Series}_1,\ \text{col}_2: \text{Series}_2,\ \ldots \}$$


In [ ]:
# Series: 1D labeled array
grades = pd.Series([88, 92, 79, 95], index=["Alice", "Bob", "Carol", "Dave"])
print(grades)
print("\nAlice's grade:", grades["Alice"])   # label-based lookup, unlike plain NumPy arrays

# DataFrame: dict of columns
df = pd.DataFrame({
    "student": ["Alice", "Bob", "Carol", "Dave"],
    "math": [88, 92, 79, 95],
    "science": [91, 85, 88, 99],
    "attendance_pct": [95, 100, 80, 90],
})
df


## 8. `loc` vs `iloc` — Label-based vs Position-based Access

- **`loc`** → select by **label/condition** (inclusive of both endpoints in slices)
- **`iloc`** → select by **integer position** (like plain NumPy, exclusive end)

Mixing these up is the #1 source of silent bugs in Pandas code.


In [ ]:
print("iloc[0] (first row by position):\n", df.iloc[0])
print("\nloc[0, 'math'] (row label 0, column 'math'):", df.loc[0, "math"])

# Conditional filtering — the Pandas equivalent of NumPy boolean masking
high_performers = df.loc[df["math"] > 85]
print("\nStudents with math > 85:\n", high_performers)


## 9. Missing Data — `isna`, `fillna`, `dropna`

Real datasets are never clean. Pandas represents missing values as `NaN` (Not a Number), and gives three core tools to handle them before any algorithm can consume the data (most ML algorithms cannot handle `NaN` natively).


In [ ]:
df_dirty = df.copy()
df_dirty.loc[1, "science"] = np.nan   # simulate a missing value
df_dirty.loc[3, "attendance_pct"] = np.nan

print("Missing value mask:\n", df_dirty.isna())
print("\nRows with any missing value dropped:\n", df_dirty.dropna())

# Mean imputation — replace NaN with the column's mean (a common, simple strategy)
df_filled = df_dirty.fillna(df_dirty.mean(numeric_only=True))
print("\nAfter mean imputation:\n", df_filled)


## 10. GroupBy — Split-Apply-Combine

`groupby` implements the **split-apply-combine** pattern: split rows into groups by a key, apply an aggregation to each group independently, then combine results into one table. Formally, for groups $G_1, ..., G_k$ partitioning the data:

$$\text{result}_k = f(\{x_i \mid x_i \in G_k\})$$


In [ ]:
df["pass_math"] = df["math"] >= 85   # derived boolean column

grouped = df.groupby("pass_math")["science"].mean()
print("Average science score, grouped by math pass/fail:\n", grouped)

# describe() gives count, mean, std, min, quartiles, max in one call — the fastest EDA sanity check
df.describe()


## 11. Mini Practical Example — NumPy + Pandas Together

We simulate a tiny "dataset" the way you'd receive one in practice: a NumPy feature matrix (from a sensor/model) that we wrap in a Pandas DataFrame for labeled exploration.


In [ ]:
np.random.seed(0)

# Simulate 100 samples, 3 features, generated as if from an experiment/sensor (NumPy's job)
raw_features = np.random.randn(100, 3) * [10, 5, 2] + [50, 20, 5]
feature_names = ["temperature", "humidity", "pressure_delta"]

# Wrap in a DataFrame for labeled inspection and EDA (Pandas' job)
sensor_df = pd.DataFrame(raw_features, columns=feature_names)
sensor_df["reading_valid"] = sensor_df["pressure_delta"] > 4  # derived label

print(sensor_df.describe())
print("\nValid vs invalid reading counts:\n", sensor_df["reading_valid"].value_counts())
print("\nMean temperature by validity:\n", sensor_df.groupby("reading_valid")["temperature"].mean())


## ✅ Key Takeaways

- NumPy arrays are fast because they're **contiguous, typed, and vectorized** — avoid Python `for` loops on numeric data whenever a vectorized alternative exists.
- **Broadcasting** and **`axis`** are the two ideas that unlock 90% of NumPy's expressive power — internalize them early.
- Pandas `Series`/`DataFrame` = NumPy arrays + labels; `loc` is label-based, `iloc` is position-based.
- Missing-value handling (`isna`/`fillna`/`dropna`) and `groupby` are non-negotiable EDA tools you'll use on every real dataset.
- Matrix multiplication (`@` / `np.dot`) is the literal operation every neural network layer performs — this notebook's linear algebra section is not incidental, it's the foundation.

**Next up:** [02_Data_Visualization](../02_Data_Visualization/) — turning these arrays and tables into plots.
